In [3]:
import torch
from transformers import AutoModelForCausalLM , AutoTokenizer
from datasets import load_dataset



In [2]:
from huggingface_hub import login
login()

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    dtype=torch.float16,
    device_map="cuda"
)

print(model)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Gem

In [4]:
model_len = len(model.model.layers)

In [5]:
model.model.layers[model_len//2]

GemmaDecoderLayer(
  (self_attn): GemmaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): GemmaMLP(
    (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
    (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
    (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
    (act_fn): GELUActivation()
  )
  (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
)

In [6]:
model.config

GemmaConfig {
  "architectures": [
    "GemmaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "dtype": "float16",
  "eos_token_id": 1,
  "head_dim": 256,
  "hidden_act": "gelu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "max_position_embeddings": 8192,
  "model_type": "gemma",
  "num_attention_heads": 8,
  "num_hidden_layers": 18,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "use_bidirectional_attention": null,
  "use_cache": true,
  "vocab_size": 256000
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2")

In [8]:
class StopForwad(Exception): pass

acts = {}
def hook(module , inp , out):
    print("Hook active")
    acts["vector"] = out[0]
    raise StopForwad


In [9]:
device = "cuda"

In [10]:


token = tokenizer("I'm a language model")
x =  torch.tensor(token.input_ids).unsqueeze(0).to(device)

In [11]:
h= model.model.layers[model_len//2].register_forward_hook(hook)

try:
    model(x)
except StopForwad:
    pass

h.remove()

Hook active


In [12]:
v =  acts["vector"] 

In [13]:
v.shape

torch.Size([7, 2048])

In [14]:
v[:1].shape

torch.Size([1, 2048])

In [15]:
from model import JumpRELUSAE

In [16]:
m = JumpRELUSAE(d_sea = 2**12)

sum(p.numel() for p in m.parameters())

18884864

In [7]:
data = load_dataset("HuggingFaceFW/fineweb-edu" ,streaming=True , split="train")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [9]:
device = "cuda"

data = data.take(2)
for text in data:
    tokens = tokenizer(text["text"], return_tensors = "pt").input_ids
    x = tokens.to(device)
    print(x.shape)

torch.Size([1, 752])
torch.Size([1, 213])


In [5]:

data = load_dataset("HuggingFaceFW/fineweb-edu" ,streaming=True , split="train")

data = data.take(1)

device = "cuda"

gemma = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b",
    dtype=torch.float16,
    device_map=device
)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

class StopForwad(Exception): pass

acts = {}
def hook(module , inp , out):
    acts["vector"] = out[0]
    raise StopForwad


h = gemma.model.layers[13].register_forward_hook(hook)

for text in data:
    token = tokenizer(text["text"] , return_tensors = "pt")
    token_ids = token.input_ids.to(device)


    with torch.no_grad():
        try:
            gemma(token_ids)
        except StopForwad:
            pass

    v = acts["vector"].float()
    

    print(v[:10])


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

tensor([[-0.5762,  0.0566,  3.0781,  ..., -0.4238,  3.4375, -0.7021],
        [-4.6875,  1.4072,  3.5039,  ...,  1.3389, -5.7070,  0.9004],
        [ 2.3027, -2.1250, -1.8076,  ..., -1.7314, -0.5898,  2.3262],
        ...,
        [ 6.3516, -2.8828, -1.7227,  ...,  0.1709, -0.8623,  0.4133],
        [ 4.3203, -1.9629,  2.9414,  ...,  1.6406, -0.6348, -1.4336],
        [ 1.2617, -1.5723,  2.1758,  ...,  1.6924,  1.7061, -1.8457]],
       device='cuda:0')


In [8]:
norms = v[:10].norm(dim=-1)
print(norms)

tensor([2135.9331,  530.4073,  161.1610,  124.9940,  147.7933,  149.8714,
         150.3732,  167.8785,  168.0411,  112.9236], device='cuda:0')


In [10]:
v[0]

tensor([-0.5762,  0.0566,  3.0781,  ..., -0.4238,  3.4375, -0.7021],
       device='cuda:0')